💡 Pour convertir ce notebook en présentation interactive, 

1. Activez le venv 

2. Lancez

   ```bash
   # sans gestionnaire de paquet
   jupyter nbconvert --to slides ISAKOV_Kiril_5_presentation_052026.ipynb
   
   # avec uv comme gestionnaire de paquet
   uv run jupyter nbconvert --to slides ISAKOV_Kiril_5_presentation_052026.ipynb
   ```

3. Ouvrez dans votre navigateur le fichier généré `ISAKOV_Kiril_5_presentation_052026.slides.html`

## « Développez une preuve de concept »

Projet n&ThinSpace;$^\text{o}$ 7 du [cursus Machine Learning Engineer][2] d'OpenClassrooms, démarré le 18/05/2026, soutenu le 06/07/2026

Étudiant : [Kiril ISAKOV][1] | Mentor : Nicolas Tisserand | Évaluateur : Mohamed Laaraiedh

Table des matières :

1. [Le dataset retenu](#/1)
2. [Les concepts de l’algorithme récent](#/2)
3. [La modélisation](#/3)
4. [Une synthèse des résultats](#/4)
5. [L’analyse de la feature importance globale et locale du nouveau modèle](#/5)
6. [Les limites et les améliorations possibles](#/6)

[1]: https://github.com/kirisakow/
[2]: https://openclassrooms.com/fr/paths/794-machine-learning-engineer

## 1. Le dataset retenu


## 1. Le dataset retenu

- Le dataset : [ImageNetDogs][1] de Stanford, composé de 20 580 images annotées et réparties entre 120 classes à raison de 150 à 250 images par classe (ce qui est plutôt bien équilibré).

- Les images au format JPEG de résolution variable, organisées en sous-dossier portant le nom de la classe correspondante, précédée d'un [WordNet ID indiquant sa catégorie synonymique][2] (ex: `n02088094-Afghan_hound`). Cette structure facilite le chargement et le prétraitement des données.

[1]: http://vision.stanford.edu/aditya86/ImageNetDogs
[2]: https://en.wikipedia.org/wiki/ImageNet#Categories


## 2. Les concepts de l’algorithme récent

## 2. Les concepts de l’algorithme récent

**YOLO (You Only Look Once)** : Approche one-stage révolutionnant la détection d'objets par sa vitesse et simplicité.

**Évolution** : YOLOv2→v3→v4→v5 (Ultralytics) avec améliorations progressives (anchor boxes, Darknet-53, CSPDarknet).

**YOLO26 (2025)** : Nouvelle génération axée sur simplicité, efficacité et déploiement edge.


## 2. Les concepts de l’algorithme récent

### Innovations clés

- **Suppression DFL** : Graphe simplifié, export multi-format facilité (+43% CPU)

- **Inférence sans NMS** : Approche end-to-end, latence réduite

- **ProgLoss & STAL** : Équilibrage adaptatif des pertes, meilleure détection des petits objets

- **Optimiseur MuSGD** : Convergence rapide et stable


## 2. Les concepts de l’algorithme récent

### Architecture unifiée

Support natif de 5 tâches : détection, segmentation, pose, détection orientée, classification.

### Optimisation edge

- Export multi-format (ONNX, TensorRT, CoreML, TFLite)

- Quantification FP16/INT8

- Latence réduite (benchmark Jetson Nano/Orin)

## 3. La modélisation


## 3. La modélisation

### Objectif

Comparaison entre un modèle dit *baseline* [EfficientNetB0][1] et un modèle dit SoTA (ou l'état de l'art) de moins de cinq ans : [YOLO26 pour la classification][2].

Le but étant d'évaluer la pertinence d'une migration vers un modèle plus récent, selon des facteurs tels que 

- précision (*accuracy*), 

- temps d'entraînement

- temps d'inférence 

- facilité de déploiement.

[1]: https://keras.io/examples/vision/image_classification_efficientnet_fine_tuning/#transfer-learning-from-pretrained-weights
[2]: https://docs.ultralytics.com/tasks/classify#where-can-i-find-pretrained-yolo26-classification-models


## 3. La modélisation

### Méthodologie commune

- **Dataset** : ImageNetDogs (120 classes, 20 580 images)

- **Split stratifié** : 80% train (16 464), 10% val (2 058), 10% test (2 058)

- **Métriques** : Accuracy, Top-5 Accuracy, F1-score macro

- **Autres indicateurs** : Temps d'entraînement/époque, temps total, temps d'inférence


## 3. La modélisation

### Modèles comparés

| Aspect | EfficientNetB0 | YOLO26 |
|--------|----------------|--------|
| **Architecture** | Compound Scaling, backbone MBConv, activation Swish ([schéma][1]) | Backbone CSPNet, blocs C3k2 avec attention spatiale, tête simplifiée ([schéma][2]) |
| **Prétraitement** | 224×224, normalisation ImageNet | Transformations internes |
| **Gestion des données** | Séquences Keras | Structure train/val/test/classe |
| **Stage 1 “Rebuild Top”** | Dégel de la tête de classification, LR=1e-3, batch=4, dropout=0.2 | Dégel “Head” + “neck”, LR=1e-3, batch=8 |
| **Stage 2 “Fine-Tuning”** | Dégel du `block7`, LR=1e-5 | Dégel “Backbone” à 2 couches sur 10, LR=1e-5 |

[1]: https://www.researchgate.net/figure/The-Architecture-of-EfficientNet-B0-a-pre-trained-deep-neural-network-designed-for-image_fig4_373342134
[2]: https://www.researchgate.net/figure/YOLO26-Architecture-Diagram_fig1_400855947

## 3. La modélisation

### Démarche d'optimisation

- **Uniformité** : mêmes données, métriques, conditions (224×224, 20 époques/stage)

- **Reproductibilité** : random seed=42

- **Focus** : LR faible pour prévenir le surapprentissage


## 3. La modélisation

### Performances temporelles

| Modèle | Entraînement | Inférence (100 images) |
|--------|--------------|------------------------|
| EfficientNetB0 | 10h09m | 4,75s |
| yolo26n-cls | 53m | 0,98s |
| yolo26s-cls | 1h03m | 1,72s |
| yolo26m-cls | 1h33m | 1,54s |

**Conclusion :** YOLO26 est jusqu'à ×10 plus rapide en entraînement, ×3-5 plus rapide en inférence, tout en maintenant une précision compétitive.


## 4. Une synthèse des résultats


## 4. Une synthèse des résultats

### Courbes d'apprentissage

- Convergence rapide pour les deux modèles

- YOLO26 : montée en précision plus abrupte grâce à son architecture optimisée et son optimiseur MuSGD. Voir les diagrammes :
  - [log/CNN__from=yolo26m-cls__n_cls=120__n_eps=20__LR=0.001__FT=2__stage1/results.png](./log/CNN__from=yolo26m-cls__n_cls=120__n_eps=20__LR=0.001__FT=2__stage1/results.png)
  - [log/CNN__from=yolo26m-cls__n_cls=120__n_eps=20__LR=1e-05__FT=2__stage2/results.png](./log/CNN__from=yolo26m-cls__n_cls=120__n_eps=20__LR=1e-05__FT=2__stage2/results.png)

- EfficientNetB0 : convergence plus lente mais performances comparables en fin d'entraînement. Pour voir les diagrammes :
  1. Lancer, depuis le venv activé `uv run tensorboard --logdir=log`
  2. Ouvrir [http://localhost:6006/](http://localhost:6006/)

## 4. Une synthèse des résultats

### Métriques de performance

| Modèle | Taille | Entraînement | Inférence (100 images) | Accuracy (top-1) | Top-5 Accuracy |
|--------|--------|--------------|------------------------|-------------------|----------------|
| EfficientNetB0 | - | 10 h 09 m | 4,75 s | - | - |
| yolo26n-cls | Nano | 53 m | 0,98 s | 77,94 % | 96,94 % |
| yolo26s-cls | Small | 1 h 03 m | 1,72 s | 84,21 % | 98,69 % |
| yolo26m-cls | Medium | 1 h 33 m | 1,54 s | 87,66 % | 99,27 % |


## 4. Une synthèse des résultats

### Comparaison et conclusion

- **Précision** : YOLO26m-cls dépasse EfficientNetB0 sur toutes les métriques, écart marqué sur le top-5 (99,27 %)

- **Efficacité** : YOLO26 est ×6-10 plus rapide en entraînement, ×3-5 plus rapide en inférence

- **Compromis** : Meilleure précision avec moins de ressources computationnelles

- **Conclusion** : La migration vers YOLO26 se justifie pleinement pour les déploiements en production, surtout sur infrastructures à ressources limitées


## 5. L’analyse de la feature importance globale et locale du nouveau modèle


## 5. L'analyse de la feature importance globale et locale du nouveau modèle

**Méthodologie :** Les couches cibles sont les **deux** dernières couches dégélées de la *backbone* du modèle `yolo26m-cls` (càd spécialisé en classification).

On distingue la feature importance **locale** (zones influentes pour chaque image individuelle) et **globale** (patterns récurrents par classe après agrégation).

**Interprétabilité :**

- L'IHM pour la prédiction et la visualisation se font via le dashboard streamlit, aussi bien en local qu'en ligne : cf [README][5]

- Les résultats de chaque inférence sont sauvegardés dans `./runs/classify/predict/nom_de_limage_sans_extension`.

- Le calcul de l'interprétabilité se fait à l'aide de la bibliothèque [EigenCAM pour YOLO26][4], plus robuste que Grad-CAM, via la superposition des cartes de chaleur basées sur la décomposition en valeurs singulières (SVD) des activations. (CAM = Class Activation Mapping)

[4]: https://github.com/rigvedrs/YOLO-26-CAM
[5]: https://github.com/kirisakow/oc-dogs-cv-dl-poc/blob/main/README.md


## 5. L'analyse de la feature importance globale et locale du nouveau modèle

### Observations

YOLO26 identifie des régions sémantiquement pertinentes :

- Races à oreilles longues (*Cocker Spaniel*, *Beagle*) : oreilles mises en évidence

- Races à fourrure distinctive (*Poodle*, *Afghan Hound*) : textures activées

**Comment ?** Deux innovations architecturales :

- Suppression de la *Non-Maximum Suppression* (NMS) : inférence de bout en bout, prédictions sans distorsion

- Suppression de la *Distribution Focal Loss* (DFL) : graphe simplifié, suivi direct du flux des caractéristiques


## 6. Les limites et les améliorations possibles


## 6. Les limites et les améliorations possibles

### Limites

- Optimisation inéquitable entre modèles

- Déséquilibres de classes non gérés

- Prétraitement différentiel entre modèles

- Interprétabilité asymétrique (EigenCAM uniquement pour YOLO26)


## 6. Les limites et les améliorations possibles

### Améliorations

#### Performances

- Optimisation hyperparamétrique systématique (GridSearch, Bayesian Optimization)
- Intégration d'augmentations avancées (CutMix, MixUp)
- Fine-tuning complet du backbone pour EfficientNetB0
- Ensemble learning combinant les deux architectures

#### Interprétabilité

- Visualisation comparative des cartes de chaleur
- Calcul de métriques quantitatives (Deletion Score, Insertion Score)
- Analyse systématique des misclassifications

#### Déploiement

- Test de quantification INT8
- Benchmark matériel (CPU, GPU, NPU)


## Fin

Des questions ?

Merci de votre attention !